<a href="https://colab.research.google.com/github/clobos/Python_Tatiana_Ecologia_Aplicada/blob/main/simulacao_modelos_series_temporais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q statsmodels plotly

In [2]:
# ============================================================
# Script em Python para simular uma série temporal, ajustar
# modelos de suavização exponencial e ARIMA, estimar parâmetros,
# produzir previsões de 10 passos e salvar gráficos em Plotly.
# ============================================================

# Importa bibliotecas para criar pastas, tratar avisos, fazer contas numéricas e trabalhar com tabelas.
import os, warnings, numpy as np, pandas as pd

# Importa os modelos de suavização exponencial: simples, com tendência e com tendência+sazonalidade.
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing

# Importa a classe ARIMA para ajustar os modelos ARIMA solicitados.
from statsmodels.tsa.arima.model import ARIMA

# Importa recursos do Plotly para criar gráficos interativos.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Oculta mensagens de aviso que não são essenciais para a demonstração.
warnings.filterwarnings('ignore')

# Cria a pasta de saída onde serão gravados os arquivos finais.
os.makedirs('output', exist_ok=True)

# Define uma semente aleatória para que a simulação seja reproduzível.
np.random.seed(123)

# Define o número total de observações simuladas.
n = 120

# Define o horizonte de previsão, neste caso 10 valores à frente.
h = 10

# Cria um índice temporal mensal para a série simulada.
idx = pd.date_range('2015-01-31', periods=n, freq='M')

# Cria uma tendência linear crescente ao longo do tempo.
trend = 0.35 * np.arange(n)

# Cria uma sazonalidade aditiva com período 12 usando função seno.
season = 8 * np.sin(2 * np.pi * np.arange(n) / 12)

# Gera um termo aleatório normal para representar ruído na série.
noise = np.random.normal(0, 2.5, n)

# Soma nível, tendência, sazonalidade e ruído para obter a série final simulada.
y = 50 + trend + season + noise

# Armazena a série simulada como um objeto pandas Series com índice temporal.
series = pd.Series(y, index=idx, name='serie')

# Cria o índice temporal correspondente aos 10 períodos futuros a serem previstos.
future_idx = pd.date_range(idx[-1] + pd.offsets.MonthEnd(1), periods=h, freq='M')

# Define um dicionário com os oito modelos pedidos pelo usuário.
models = {
    '1_SES_sem_tend_sem_sazon': ('ses', {}),
    '2_Holt_com_tend_sem_sazon': ('holt', {}),
    '3_HoltWinters_tend_sazon': ('hw', {'trend':'add', 'seasonal':'add', 'seasonal_periods':12}),
    '4_ARIMA_2_0_0': ('arima', {'order': (2,0,0)}),
    '5_ARIMA_2_1_0': ('arima', {'order': (2,1,0)}),
    '6_ARIMA_0_0_2': ('arima', {'order': (0,0,2)}),
    '7_ARIMA_0_1_2': ('arima', {'order': (0,1,2)}),
    '8_ARIMA_2_2_2': ('arima', {'order': (2,2,2)}),
}

# Cria listas vazias para armazenar parâmetros estimados, previsões e resumo dos modelos.
param_rows = []
forecast_rows = []
summary_rows = []

# Inicia um laço para ajustar cada modelo definido anteriormente.
for nome, spec in models.items():

    # Separa o tipo do modelo e seus argumentos específicos.
    kind, kwargs = spec

    # Verifica se o modelo atual é de suavização exponencial simples.
    if kind == 'ses':

        # Ajusta o modelo SES à série simulada estimando os valores iniciais.
        fit = SimpleExpSmoothing(series, initialization_method='estimated').fit()

        # Gera previsões de 10 passos à frente.
        forecast = fit.forecast(h)

        # Captura os parâmetros estimados pelo modelo ajustado.
        params = fit.params

        # Organiza os parâmetros em um dicionário padronizado.
        param_map = {
            'smoothing_level': params.get('smoothing_level'),
            'smoothing_trend': params.get('smoothing_trend'),
            'smoothing_seasonal': params.get('smoothing_seasonal'),
            'damping_trend': params.get('damping_trend'),
            'initial_level': params.get('initial_level'),
            'initial_trend': params.get('initial_trend'),
            'sse': fit.sse,
            'aic': getattr(fit, 'aic', np.nan),
            'bic': getattr(fit, 'bic', np.nan)
        }

    # Verifica se o modelo atual é o de Holt com tendência e sem sazonalidade.
    elif kind == 'holt':

        # Ajusta o modelo de Holt à série.
        fit = Holt(series, initialization_method='estimated').fit()

        # Calcula as previsões futuras.
        forecast = fit.forecast(h)

        # Obtém os parâmetros estimados do ajuste.
        params = fit.params

        # Organiza os parâmetros em um dicionário.
        param_map = {
            'smoothing_level': params.get('smoothing_level'),
            'smoothing_trend': params.get('smoothing_trend'),
            'smoothing_seasonal': params.get('smoothing_seasonal'),
            'damping_trend': params.get('damping_trend'),
            'initial_level': params.get('initial_level'),
            'initial_trend': params.get('initial_trend'),
            'sse': fit.sse,
            'aic': getattr(fit, 'aic', np.nan),
            'bic': getattr(fit, 'bic', np.nan)
        }

    # Verifica se o modelo atual é Holt-Winters com tendência e sazonalidade aditivas.
    elif kind == 'hw':

        # Ajusta o modelo Holt-Winters à série.
        fit = ExponentialSmoothing(
            series,
            trend=kwargs['trend'],
            seasonal=kwargs['seasonal'],
            seasonal_periods=kwargs['seasonal_periods'],
            initialization_method='estimated'
        ).fit()

        # Calcula as previsões futuras.
        forecast = fit.forecast(h)

        # Obtém os parâmetros estimados do ajuste.
        params = fit.params

        # Organiza os parâmetros em um dicionário.
        param_map = {
            'smoothing_level': params.get('smoothing_level'),
            'smoothing_trend': params.get('smoothing_trend'),
            'smoothing_seasonal': params.get('smoothing_seasonal'),
            'damping_trend': params.get('damping_trend'),
            'initial_level': params.get('initial_level'),
            'initial_trend': params.get('initial_trend'),
            'sse': fit.sse,
            'aic': getattr(fit, 'aic', np.nan),
            'bic': getattr(fit, 'bic', np.nan)
        }

    # Caso contrário, trata-se de um modelo ARIMA.
    else:

        # Ajusta o modelo ARIMA definido em kwargs['order'] à série simulada.
        fit = ARIMA(series, order=kwargs['order']).fit()

        # Gera as previsões de 10 passos à frente.
        forecast = fit.forecast(h)

        # Transforma os parâmetros do modelo em dicionário.
        param_map = dict(fit.params)

        # Adiciona AIC e BIC ao dicionário de parâmetros.
        param_map['aic'] = fit.aic
        param_map['bic'] = fit.bic

        # Calcula a soma de quadrados dos resíduos como medida de ajuste.
        param_map['sse'] = np.sum(np.square(fit.resid))

    # Cria uma linha identificando o nome do modelo.
    row = {'modelo': nome}

    # Junta o nome do modelo com seus parâmetros estimados.
    row.update(param_map)

    # Adiciona essa linha à lista final de parâmetros.
    param_rows.append(row)

    # Armazena, um a um, os 10 valores previstos para o modelo atual.
    for i, val in enumerate(forecast, start=1):
        forecast_rows.append({
            'modelo': nome,
            'passo': i,
            'data_prevista': future_idx[i-1],
            'previsao': float(val)
        })

    # Guarda um pequeno resumo numérico do desempenho do modelo.
    summary_rows.append({
        'modelo': nome,
        'previsao_1': float(forecast.iloc[0]),
        'previsao_10': float(forecast.iloc[-1]),
        'aic': row.get('aic', np.nan),
        'bic': row.get('bic', np.nan)
    })

    # Cria um gráfico apenas com a série original.
    fig1 = go.Figure()

    # Adiciona a linha da série observada ao gráfico.
    fig1.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines',
        name='Série original',
        line=dict(color='#1f77b4')
    ))

    # Define título e rótulos dos eixos do gráfico da série original.
    fig1.update_layout(
        title=f'{nome} - Série original',
        xaxis_title='Tempo',
        yaxis_title='Valor',
        template='plotly_white'
    )

    # Salva o gráfico interativo da série original em HTML.
    fig1.write_html(f'output/{nome}_serie_original.html', include_plotlyjs='cdn')

    # Cria um novo gráfico para mostrar série original e previsão juntas.
    fig2 = go.Figure()

    # Adiciona a série observada ao gráfico combinado.
    fig2.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines',
        name='Série original',
        line=dict(color='#1f77b4')
    ))

    # Adiciona os 10 valores previstos ao gráfico combinado.
    fig2.add_trace(go.Scatter(
        x=future_idx,
        y=forecast.values,
        mode='lines+markers',
        name='Previsão (10)',
        line=dict(color='#d62728', dash='dash')
    ))

    # Define layout do gráfico com previsões.
    fig2.update_layout(
        title=f'{nome} - Série original com 10 previsões',
        xaxis_title='Tempo',
        yaxis_title='Valor',
        template='plotly_white'
    )

    # Salva o gráfico interativo da série com previsão em HTML.
    fig2.write_html(f'output/{nome}_serie_com_previsao.html', include_plotlyjs='cdn')

# Converte a lista de parâmetros em tabela pandas.
param_table = pd.DataFrame(param_rows)

# Converte a lista de previsões em tabela pandas.
forecast_table = pd.DataFrame(forecast_rows)

# Converte o resumo dos modelos em tabela pandas.
summary_table = pd.DataFrame(summary_rows)

# Salva a tabela com estimativas dos parâmetros em arquivo CSV.
param_table.to_csv('output/tabela_parametros_modelos.csv', index=False)

# Salva a tabela com as previsões de 10 passos em arquivo CSV.
forecast_table.to_csv('output/tabela_previsoes_10_passos.csv', index=False)

# Salva um resumo geral dos modelos em arquivo CSV.
summary_table.to_csv('output/resumo_modelos.csv', index=False)

# Salva a série simulada original em arquivo CSV.
series.to_csv('output/serie_simulada.csv', header=True)

# Cria uma figura com subgráficos para comparar todos os modelos de uma vez.
fig_all = make_subplots(rows=4, cols=2, subplot_titles=list(models.keys()))

# Percorre os modelos para inserir cada série e previsão no painel comparativo.
for j, nome in enumerate(models.keys(), start=1):

    # Determina a linha do subgráfico.
    r = (j-1)//2 + 1

    # Determina a coluna do subgráfico.
    c = (j-1)%2 + 1

    # Filtra as previsões correspondentes ao modelo atual.
    prev = forecast_table[forecast_table['modelo'] == nome]

    # Adiciona a série original ao subgráfico.
    fig_all.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines',
        line=dict(color='#1f77b4'),
        name='Série original',
        showlegend=(j == 1)
    ), row=r, col=c)

    # Adiciona a previsão ao subgráfico.
    fig_all.add_trace(go.Scatter(
        x=prev['data_prevista'],
        y=prev['previsao'],
        mode='lines+markers',
        line=dict(color='#d62728', dash='dash'),
        name='Previsão',
        showlegend=(j == 1)
    ), row=r, col=c)

# Ajusta o layout do painel geral comparativo.
fig_all.update_layout(
    height=1200,
    width=1000,
    title='Série original e previsões de 10 passos para todos os modelos',
    template='plotly_white'
)

# Salva o painel comparativo em HTML interativo.
fig_all.write_html('output/painel_geral_modelos.html', include_plotlyjs='cdn')

# Mostra as tabelas principais no Colab.
print("\nTabela de parâmetros estimados:")
display(param_table)

print("\nTabela de previsões:")
display(forecast_table)

print("\nResumo dos modelos:")
display(summary_table)

# Exibe o painel geral no notebook.
fig_all.show()


Tabela de parâmetros estimados:


,modelo,smoothing_level,smoothing_trend,smoothing_seasonal,damping_trend,initial_level,initial_trend,sse,aic,bic,const,ar.L1,ar.L2,sigma2,ma.L1,ma.L2
0,1_SES_sem_tend_sem_sazon,9.347368e-01,NaN,NaN,NaN,47.915183,NaN,2898.425082,386.131726,391.706710,NaN,NaN,NaN,NaN,NaN,NaN
1,2_Holt_com_tend_sem_sazon,9.312146e-01,0.000000e+00,NaN,NaN,47.632560,0.295014,2886.536930,389.638523,400.788490,NaN,NaN,NaN,NaN,NaN,NaN
2,3_HoltWinters_tend_sazon,1.490116e-08,8.692919e-09,0.0,NaN,49.888819,0.347853,911.037731,275.251109,319.850976,NaN,NaN,NaN,NaN,NaN,NaN
3,4_ARIMA_2_0_0,NaN,NaN,NaN,NaN,NaN,NaN,3286.110110,730.587050,741.737017,69.691224,0.896609,0.041876,23.719295,NaN,NaN
4,5_ARIMA_2_1_0,NaN,NaN,NaN,NaN,NaN,NaN,5123.799050,723.192993,731.530363,NaN,-0.070736,0.058103,24.260373,NaN,NaN
5,6_ARIMA_0_0_2,NaN,NaN,NaN,NaN,NaN,NaN,6241.026282,817.859478,829.009445,70.684628,NaN,NaN,49.484302,0.923386,0.531272
6,7_ARIMA_0_1_2,NaN,NaN,NaN,NaN,NaN,NaN,5084.359541,721.418305,729.755676,NaN,NaN,NaN,23.880514,-0.149549,0.221858
7,8_ARIMA_2_2_2,NaN,NaN,NaN,NaN,NaN,NaN,5862.752934,741.666763,755.520186,NaN,-1.034966,-0.468764,28.660104,0.236627,-0.176441



Tabela de previsões:


,modelo,passo,data_prevista,previsao
0,1_SES_sem_tend_sem_sazon,1,2025-01-31,82.992683
1,1_SES_sem_tend_sem_sazon,2,2025-02-28,82.992683
2,1_SES_sem_tend_sem_sazon,3,2025-03-31,82.992683
3,1_SES_sem_tend_sem_sazon,4,2025-04-30,82.992683
4,1_SES_sem_tend_sem_sazon,5,2025-05-31,82.992683
...,...,...,...,...
75,8_ARIMA_2_2_2,6,2025-06-30,82.814575
76,8_ARIMA_2_2_2,7,2025-07-31,82.839802
77,8_ARIMA_2_2_2,8,2025-08-31,82.227312
78,8_ARIMA_2_2_2,9,2025-09-30,82.141562



Resumo dos modelos:


,modelo,previsao_1,previsao_10,aic,bic
0,1_SES_sem_tend_sem_sazon,82.992683,82.992683,386.131726,391.706710
1,2_Holt_com_tend_sem_sazon,83.329592,85.984715,389.638523,400.788490
2,3_HoltWinters_tend_sazon,91.740168,87.532034,275.251109,319.850976
3,4_ARIMA_2_0_0,82.073926,76.855448,730.587050,741.737017
4,5_ARIMA_2_1_0,83.475630,83.081094,723.192993,731.530363
5,6_ARIMA_0_0_2,74.984904,70.684628,817.859478,829.009445
6,7_ARIMA_0_1_2,85.456566,84.176653,721.418305,729.755676
7,8_ARIMA_2_2_2,84.417146,81.809592,741.666763,755.520186
